In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

/Users/martindufour/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Function 8
print('Function 8')
func8_inputs = np.load('./initial_data/function_8/initial_inputs.npy')
print(func8_inputs)

func8_outputs = np.load('./initial_data/function_8/initial_outputs.npy')
print(func8_outputs)
print('/n')

Function 8
[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.

In [3]:
# Week 1 Input and Output Data
week_1_inputs = [ np.array([0.155793, 0.528435]), np.array([0.224164, 0.812385]), np.array([0.830919, 0.158523, 0.528406]), np.array([0.912533, 0.052672, 0.771239, 0.219812]), np.array([0.234189, 0.83648 , 0.884484, 0.873516]), np.array([0.490808, 0.618683, 0.277824, 0.900494, 0.106596]), np.array([0.067896, 0.486672, 0.255422, 0.215118, 0.427428, 0.72097 ]), np.array([0.061447, 0.062956, 0.029929, 0.036786, 0.407935, 0.795055, 0.496307,0.888085]) ]
week_1_outputs = [np.float64(1.5311489892413605e-58), np.float64(0.04614596805685454), np.float64(-0.04591165123945737), np.float64(-24.387232512869755), np.float64(1049.4420694211206), np.float64(-0.849252655626155), np.float64(1.3793294734939503), np.float64(9.598780741169)]

week_2_inputs = [np.array([0.946399, 0.18731 ]), np.array([0.712637, 0.921564]), np.array([0.765104, 0.052672, 0.438597]), np.array([0.380965, 0.771701, 0.089479, 0.536968]), np.array([0.219189, 0.85148 , 0.874484, 0.883516]), np.array([0.114755, 0.697421, 0.354179, 0.887624, 0.589139]), np.array([0.070896, 0.484672, 0.259422, 0.216118, 0.427428, 0.72297 ]), np.array([0.062447, 0.061956, 0.030929, 0.035786, 0.408935, 0.794055, 0.497307, 0.887085])]
week_2_outputs = [np.float64(2.338449488843798e-206), np.float64(0.5728372778652475), np.float64(-0.10235480701941752), np.float64(-13.368584815829887), np.float64(1109.9883069580462), np.float64(-1.492909868264592), np.float64(1.3944037721068683), np.float64(9.598978827169)]

week_3_inputs = [np.array([0.583738, 0.706798]), np.array([0.699637, 0.928564]), np.array([0.123457, 0.876543, 0.5     ]), np.array([0.230785, 0.914568, 0.102938, 0.657322]), np.array([0.214189, 0.85648 , 0.869484, 0.888516]), np.array([0.051235, 0.987654, 0.43211 , 0.123457, 0.765432]), np.array([0.071896, 0.483672, 0.261422, 0.217118, 0.426428, 0.72497 ]), np.array([0.063447, 0.060956, 0.031929, 0.034786, 0.409935, 0.793055, 0.498307, 0.886085])]
week_3_outputs = [np.float64(9.817718646044271e-07), np.float64(0.49978778689465564), np.float64(-0.04054152149606349), np.float64(-21.112987453792396), np.float64(1132.5255136709882), np.float64(-2.4679805862566795), np.float64(1.4059619293543582), np.float64(9.599155713169)]

week_4_inputs = [np.array([0.867072, 0.913241]), np.array([0.83604 , 0.696071]), np.array([0.447658, 0.395195, 0.505344]), np.array([0.422706, 0.385497, 0.37529 , 0.410833]), np.array([0.157706, 0.912326, 0.830158, 0.925715]), np.array([0.275401, 0.      , 0.568079, 1.      , 0.121326]), np.array([0.124679, 0.389027, 0.389012, 0.23686 , 0.379036, 0.802247]), np.array([0.077447, 0.21029 , 0.114032, 0.159535, 0.695924, 0.531316,0.178973, 0.57168 ])]
week_4_outputs = [np.float64(-1.0508862613282513e-96), np.float64(0.20657541488475104), np.float64(-0.03229682155733878), np.float64(0.4964978124935766), np.float64(1445.380906735266), np.float64(-0.8152779914672599), np.float64(2.0217812896063525), np.float64(9.987130922543)]

week_5_inputs = [np.array([0.065052, 0.948886]), np.array([0.388677, 0.271349]), np.array([1.      , 0.136717, 0.850593]), np.array([0.908266, 0.239562, 0.144895, 0.489453]), np.array([0.115614, 0.955641, 0.820859, 0.938174]), np.array([0.568442, 0.      , 1.      , 1.      , 1.      ]), np.array([0.134015, 0.028783, 0.755137, 0.62031 , 0.70408 , 0.212964]), np.array([0.127008, 0.292876, 0.06967 , 0.277582, 0.553407, 0.547258,0.220835, 0.443146])]
week_5_outputs = [np.float64(2.0262778967114778e-283), np.float64(0.016418658339648333), np.float64(-0.054388754089278846), np.float64(-17.161465002411145), np.float64(1779.8600577462366), np.float64(-1.9906490554141107), np.float64(0.052467603616080494), np.float64(9.9149654065019)]

week_6_inputs = [np.array([0.17701 , 0.088703]), np.array([0.593592, 0.679102]), np.array([0.7536, 1.    , 1.    ]), np.array([0.370348, 0.379959, 0.430216, 0.444351]), np.array([0.169493, 0.556801, 0.936155, 0.69603 ]), np.array([0.366464, 0.316099, 1.      , 1.      , 0.      ]), np.array([0.123574, 0.270452, 0.482301, 0.208163, 0.330398, 0.872733]), np.array([0.3191  , 0.828915, 0.037008, 0.59627 , 0.230009, 0.120567,0.076953, 0.696289])]
week_6_outputs = [np.float64(-2.3725238219366144e-119), np.float64(0.06836721478932847), np.float64(-0.48310415434111403), np.float64(0.18076540708623456), np.float64(282.83820524691396), np.float64(-0.8214281898088153), np.float64(2.075605759888563), np.float64(8.8803427965834)]

week_7_inputs = [np.array([0.065052, 0.948886]), np.array([0.140924, 0.802197]), np.array([1., 1., 0.]), np.array([0.32078 , 0.186519, 0.040775, 0.590893]), np.array([0.548734, 0.691895, 0.651961, 0.224269]), np.array([0.368433, 0.      , 1.      , 1.      , 0.428022]), np.array([0.139689, 0.317433, 0.463608, 0.250431, 0.325485, 0.810756]), np.array([0.      , 0.202823, 0.231703, 0.      , 1.      , 1.      ,
       0.288474, 1.      ])]
week_7_outputs = [np.float64(2.0262778967114778e-283), np.float64(-0.10826299524356352), np.float64(-0.16354530625442043), np.float64(-11.58523458824008), np.float64(1.9931553503870212), np.float64(-1.2796687884296385), np.float64(2.462201676843452), np.float64(9.622023932692)]

week_8_inputs = [np.array([0.000788, 0.033717]), np.array([0.914607, 0.789979]), np.array([0.317253, 0.002183, 0.963506]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.078217, 0.973099, 0.868006, 0.933352]), np.array([0.426158, 0.348959, 0.616644, 0.692851, 0.024814]), np.array([0.269889, 0.346609, 0.467584, 0.256632, 0.302654, 0.800092]), np.array([0.066269, 0.029193, 0.143059, 0.209133, 0.840619, 0.604919,
       0.230297, 0.701468])]
week_8_outputs = [np.float64(1.5608341712501477e-228), np.float64(0.0347797753016137), np.float64(-0.3756702789549372), np.float64(-33.661790988299735), np.float64(2151.3700669834334), np.float64(-0.23000336822278494), np.float64(2.4516632535923746), np.float64(9.9644234438351)]

week_9_inputs = [np.array([0.000924, 0.003116]), np.array([0.914607, 0.789979]), np.array([0.047574, 0.998325, 0.999125]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.080698, 0.971918, 0.842327, 0.954471]), np.array([0.942348, 0.037756, 0.105925, 0.010348, 0.997595]), np.array([0.090198, 0.680559, 0.851748, 0.080076, 0.298474, 0.701193]), np.array([0.95983 , 0.002073, 0.011779, 0.227207, 0.571611, 0.031204,
       0.24468 , 0.055697])]
week_9_outputs = [np.float64(7.25285761175276e-246), np.float64(0.05670257386671321), np.float64(-0.48158498276260003), np.float64(-33.661790988299735), np.float64(2156.522419821577), np.float64(-3.026993482898997), np.float64(0.6117443854593981), np.float64(8.1721431718416)]

week_10_inputs = [np.array([0.999878, 0.001995]), np.array([0.41611 , 0.666998]), np.array([0.001378, 0.999625, 0.642937]), np.array([0.027746, 0.654937, 0.999652, 0.024674]), np.array([0.109191, 0.976767, 0.866838, 0.873854]), np.array([0.944032, 0.013284, 0.976966, 0.023315, 0.997903]), np.array([0.167912, 0.257921, 0.480203, 0.472529, 0.254657, 0.778833]), np.array([0.116951, 0.002708, 0.212486, 0.121597, 0.986678, 0.488014,
       0.152731, 0.379341])]
week_10_outputs = [np.float64(0.0), np.float64(0.1625184332362701), np.float64(-0.13457392031599885), np.float64(-30.133085064011215), np.float64(1769.6901405375768), np.float64(-2.7724961974553874), np.float64(1.9208154093840464), np.float64(9.9296060847489)]

week_11_inputs = [np.array([0.37454 , 0.950714]), np.array([0.41611 , 0.666998]), np.array([0.997078, 0.475121, 0.651523]), np.array([0.00109 , 0.902069, 0.972212, 0.16754 ]), np.array([0.019324, 0.941663, 0.833048, 0.950636]), np.array([0.973398, 0.011482, 0.130865, 0.931063, 0.997256]), np.array([0.215391, 0.288991, 0.602008, 0.322698, 0.265768, 0.811541]), np.array([0.16109 , 0.049964, 0.210829, 0.130137, 0.965223, 0.408414,
       0.113403, 0.289162])]
week_11_outputs = [np.float64(-1.560646704467778e-117), np.float64(-0.1334547156009971), np.float64(-0.10208280924057045), np.float64(-31.74483921038956), np.float64(1827.9676185989063), np.float64(-2.432448557520746), np.float64(2.4819294367786457), np.float64(9.9158368797091)]

In [4]:
# Function 8
print('Function 8')
# Load inputs from previous run
# Loads initial data
week_0_func8_inputs = np.load('./initial_data/function_8/initial_inputs.npy')
week_0_func8_outputs = np.load('./initial_data/function_8/initial_outputs.npy')

print(f'Shape of initial input data: {week_0_func8_inputs.shape}')
print(f'Shape of initial output data: {week_0_func8_outputs.shape}')

print(f'Week 1 inputs: {week_1_inputs[7]}')
print(f'Week 2 inputs: {week_2_inputs[7]}')
print(f'Week 3 inputs: {week_3_inputs[7]}')
print(f'Week 4 inputs: {week_4_inputs[7]}')
print(f'Week 5 inputs: {week_5_inputs[7]}')
print(f'Week 6 inputs: {week_6_inputs[7]}')
print(f'Week 7 inputs: {week_7_inputs[7]}')
print(f'Week 8 inputs: {week_8_inputs[7]}')
print(f'Week 9 inputs: {week_9_inputs[7]}')
print(f'Week 10 inputs: {week_10_inputs[7]}')
print(f'Week 11 inputs: {week_11_inputs[7]}')

combined_func8_inputs = np.vstack([
    week_0_func8_inputs,
    week_1_inputs[7],
    week_2_inputs[7],
    week_3_inputs[7],
    week_4_inputs[7],
    week_5_inputs[7],
    week_6_inputs[7],
    week_7_inputs[7],
    week_8_inputs[7],
    week_9_inputs[7],
    week_10_inputs[7],
    week_11_inputs[7]
])
print(f'Number of input data points: {len(combined_func8_inputs)}')
print('Combined input data')
print(combined_func8_inputs)

# Load outputs from previous run
week_func8_output = week_1_outputs[7]
combined_func8_outputs = np.concatenate([
    week_0_func8_outputs,
    [week_1_outputs[7]],
    [week_2_outputs[7]],
    [week_3_outputs[7]],
    [week_4_outputs[7]],
    [week_5_outputs[7]],
    [week_6_outputs[7]],
    [week_7_outputs[7]],
    [week_8_outputs[7]],
    [week_9_outputs[7]],
    [week_10_outputs[7]],
    [week_11_outputs[7]]
])
print(f'Number of output data points: {len(combined_func8_outputs)}')
print('Combined output data')
print(combined_func8_outputs)

Function 8
Shape of initial input data: (40, 8)
Shape of initial output data: (40,)
Week 1 inputs: [0.061447 0.062956 0.029929 0.036786 0.407935 0.795055 0.496307 0.888085]
Week 2 inputs: [0.062447 0.061956 0.030929 0.035786 0.408935 0.794055 0.497307 0.887085]
Week 3 inputs: [0.063447 0.060956 0.031929 0.034786 0.409935 0.793055 0.498307 0.886085]
Week 4 inputs: [0.077447 0.21029  0.114032 0.159535 0.695924 0.531316 0.178973 0.57168 ]
Week 5 inputs: [0.127008 0.292876 0.06967  0.277582 0.553407 0.547258 0.220835 0.443146]
Week 6 inputs: [0.3191   0.828915 0.037008 0.59627  0.230009 0.120567 0.076953 0.696289]
Week 7 inputs: [0.       0.202823 0.231703 0.       1.       1.       0.288474 1.      ]
Week 8 inputs: [0.066269 0.029193 0.143059 0.209133 0.840619 0.604919 0.230297 0.701468]
Week 9 inputs: [0.95983  0.002073 0.011779 0.227207 0.571611 0.031204 0.24468  0.055697]
Week 10 inputs: [0.116951 0.002708 0.212486 0.121597 0.986678 0.488014 0.152731 0.379341]
Week 11 inputs: [0.16109 

In [5]:
### ====== OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 8 ======
# Import the OptunaBayesianOptimizer class
import sys
sys.path.insert(0, './bayesian_optimization_challenge-md')
from bo_optuna import OptunaBayesianOptimizer

# Create optimizer instance with initial Function 8 data
print("=" * 60)
print("OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 8")
print("=" * 60)

X_func8_initial = week_0_func8_inputs
y_func8_initial = week_0_func8_outputs
bounds_func8 = [(0, 1)] * X_func8_initial.shape[1]

optimizer_func8 = OptunaBayesianOptimizer(
    X_initial=X_func8_initial,
    y_initial=y_func8_initial,
    bounds=bounds_func8,
    optimize_hp=True,  # Enable hyperparameter tuning
    random_state=42,
    acquisition="ei"
)

print(f"\nInitial training data shape: X={optimizer_func8.X_train.shape}, y={optimizer_func8.y_train.shape}")
print(f"Initial best observation: {optimizer_func8.get_best_observation()[1]:.6e}")


OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 8

Initial training data shape: X=(40, 8), y=(40,)
Initial best observation: 9.598482e+00


In [6]:
# Run Optuna-based BO for 8 weeks (8 iterations)
print("\nRunning Optuna-based BO for 8 iterations...")
print("-" * 60)

# Get all weekly data
weekly_data = [
    (week_1_inputs[7], week_1_outputs[7]),
    (week_2_inputs[7], week_2_outputs[7]),
    (week_3_inputs[7], week_3_outputs[7]),
    (week_4_inputs[7], week_4_outputs[7]),
    (week_5_inputs[7], week_5_outputs[7]),
    (week_6_inputs[7], week_6_outputs[7]),
    (week_7_inputs[7], week_7_outputs[7]),
    (week_8_inputs[7], week_8_outputs[7]),
    (week_9_inputs[7], week_9_outputs[7]),
    (week_10_inputs[7], week_10_outputs[7]),
    (week_11_inputs[7], week_11_outputs[7])
]

optuna_proposals_func8 = []
manual_best_func8 = week_0_func8_outputs.max()
optuna_best_func8 = y_func8_initial.max()

for week, (x_actual, y_actual) in enumerate(weekly_data, start=1):
    print(f"\nWeek {week}:")
    print(f"  Actual observation: y = {y_actual:.6e}")
    
    # Get Optuna proposal
    proposals = optimizer_func8.optimize(
        n_iterations=1,
        optimize_hp_every=1 if week % 2 == 0 else 0,  # Tune HP every other week
        optimize_hp_n_trials=30,
        acq_n_trials=100,
        verbose=True
    )
    
    x_proposed = proposals[0]
    optuna_proposals_func8.append(x_proposed)
    
    # Update optimizer with actual observation
    optimizer_func8.update(x_actual, y_actual)
    
    # Track best values
    manual_best_func8 = max(manual_best_func8, y_actual)
    optuna_best_func8 = max(optuna_best_func8, y_actual)
    
    print(f"  Best so far (Optuna): {optuna_best_func8:.6e}")

print("\n" + "=" * 60)
print("OPTUNA-BASED BO COMPLETED FOR FUNCTION 8")
print("=" * 60)



Running Optuna-based BO for 8 iterations...
------------------------------------------------------------

Week 1:
  Actual observation: y = 9.598781e+00


[I 2026-04-26 09:11:09,581] A new study created in memory with name: no-name-6cc9be80-ba37-4a61-bb5d-9b1bfaab4ca9
[I 2026-04-26 09:11:09,584] Trial 0 finished with value: 4.704960500602823e-48 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051, 'x3': 0.5986584841970366, 'x4': 0.15601864044243652, 'x5': 0.15599452033620265, 'x6': 0.05808361216819946, 'x7': 0.8661761457749352}. Best is trial 0 with value: 4.704960500602823e-48.
[I 2026-04-26 09:11:09,586] Trial 1 finished with value: 0.00011281240694951998 and parameters: {'x0': 0.6011150117432088, 'x1': 0.7080725777960455, 'x2': 0.020584494295802447, 'x3': 0.9699098521619943, 'x4': 0.8324426408004217, 'x5': 0.21233911067827616, 'x6': 0.18182496720710062, 'x7': 0.18340450985343382}. Best is trial 1 with value: 0.00011281240694951998.
[I 2026-04-26 09:11:09,589] Trial 2 finished with value: 0.00019640633558887233 and parameters: {'x0': 0.3042422429595377, 'x1': 0.5247564316322378, 'x2': 0.4319450

[Iteration 0] Proposed: [0.03993619 0.31786408 0.170237   0.24955376 0.65478177 0.64141606
 0.11950684 0.66495127], EI: 0.460636
  Best so far (Optuna): 9.598781e+00

Week 2:
  Actual observation: y = 9.598979e+00


[I 2026-04-26 09:11:11,578] Trial 10 finished with value: 1.7766820616611327e-41 and parameters: {'x0': 0.9451365442303865, 'x1': 0.005997182955817526, 'x2': 0.3687287727210974, 'x3': 0.0179618756148196, 'x4': 0.2726868772010844, 'x5': 0.6902470735691453, 'x6': 0.9597707459454199, 'x7': 0.6765723797427797}. Best is trial 5 with value: 0.006538635898184042.
[I 2026-04-26 09:11:11,602] Trial 11 finished with value: 0.03245849950289042 and parameters: {'x0': 0.2373788740346892, 'x1': 0.5026361331407643, 'x2': 0.3962212828105071, 'x3': 0.2695873589446723, 'x4': 0.586135836689225, 'x5': 0.4688790033325413, 'x6': 0.34777505392728825, 'x7': 0.6195339431335991}. Best is trial 11 with value: 0.03245849950289042.
[I 2026-04-26 09:11:11,716] Trial 12 finished with value: 0.21287125974711357 and parameters: {'x0': 0.20003379284295986, 'x1': 0.30179874756539615, 'x2': 0.28174565681996244, 'x3': 0.2143716355988109, 'x4': 0.4894028978010987, 'x5': 0.4632899268170014, 'x6': 0.4321051437594004, 'x7': 0

[Iteration 0] Proposed: [0.02306934 0.13878792 0.14086556 0.21501341 0.73948348 0.56698148
 0.13384321 0.5011429 ], EI: 0.494022
  Best so far (Optuna): 9.598979e+00

Week 3:
  Actual observation: y = 9.599156e+00


...

[Iteration 0] Proposed: [0.11695125 0.00270795 0.21248597 0.12159731 0.98667772 0.48801352
 0.15273121 0.37934123], EI: 0.003146
  Best so far (Optuna): 9.987131e+00

Week 10:
  Actual observation: y = 9.929606e+00


[I 2026-04-26 09:11:26,696] A new study created in memory with name: no-name-cf9184fc-28b7-461a-8281-4f1eb6eb44e6
[I 2026-04-26 09:11:26,699] Trial 0 finished with value: 4.550886583834869e-104 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051, 'x3': 0.5986584841970366, 'x4': 0.15601864044243652, 'x5': 0.15599452033620265, 'x6': 0.05808361216819946, 'x7': 0.8661761457749352}. Best is trial 0 with value: 4.550886583834869e-104.
[I 2026-04-26 09:11:26,701] Trial 1 finished with value: 2.914359550197521e-10 and parameters: {'x0': 0.6011150117432088, 'x1': 0.7080725777960455, 'x2': 0.020584494295802447, 'x3': 0.9699098521619943, 'x4': 0.8324426408004217, 'x5': 0.21233911067827616, 'x6': 0.18182496720710062, 'x7': 0.18340450985343382}. Best is trial 1 with value: 2.914359550197521e-10.
[I 2026-04-26 09:11:26,704] Trial 2 finished with value: 3.762761465914589e-35 and parameters: {'x0': 0.3042422429595377, 'x1': 0.5247564316322378, 'x2': 0.43194501

[Iteration 0] Proposed: [0.16108951 0.04996444 0.21082928 0.13013739 0.96522255 0.40841433
 0.1134031  0.28916155], EI: 0.001619
  Best so far (Optuna): 9.987131e+00

Week 11:
  Actual observation: y = 9.915837e+00


[I 2026-04-26 09:11:28,518] A new study created in memory with name: no-name-093cd486-be53-464f-b0fb-d46c37c001fd
[I 2026-04-26 09:11:28,520] Trial 0 finished with value: 2.7207157528940685e-106 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051, 'x3': 0.5986584841970366, 'x4': 0.15601864044243652, 'x5': 0.15599452033620265, 'x6': 0.05808361216819946, 'x7': 0.8661761457749352}. Best is trial 0 with value: 2.7207157528940685e-106.
[I 2026-04-26 09:11:28,522] Trial 1 finished with value: 5.448412510516765e-10 and parameters: {'x0': 0.6011150117432088, 'x1': 0.7080725777960455, 'x2': 0.020584494295802447, 'x3': 0.9699098521619943, 'x4': 0.8324426408004217, 'x5': 0.21233911067827616, 'x6': 0.18182496720710062, 'x7': 0.18340450985343382}. Best is trial 1 with value: 5.448412510516765e-10.
[I 2026-04-26 09:11:28,524] Trial 2 finished with value: 3.116977246344479e-36 and parameters: {'x0': 0.3042422429595377, 'x1': 0.5247564316322378, 'x2': 0.431945

[Iteration 0] Proposed: [4.18176946e-04 1.05124306e-02 1.95004505e-01 2.11749938e-01
 7.88533382e-01 9.00374023e-01 2.41123024e-01 3.34541202e-01], EI: 0.001023
  Best so far (Optuna): 9.987131e+00

OPTUNA-BASED BO COMPLETED FOR FUNCTION 8
